# Split-Plot Designs

## Overview
Split-plot designs arise when one factor is applied to large experimental units (whole plots) and a second factor is applied to sub-units within each whole plot. They produce a **partly nested** structure with two distinct error terms.

**Classic ecology example:** disturbance treatment applied to whole plots; species addition applied to subplots within each plot. The whole-plot error (variation among plots within disturbance treatments) is separate from the subplot error.

**Why two error terms?**
Whole-plot treatments are replicated at the whole-plot level — the error for testing them is whole-plot variation. Subplot treatments and interactions are tested against the subplot error.

**Quinn & Keough (2002) ch. 11.**

---

In [ ]:
library(tidyverse); library(lme4); library(lmerTest); library(emmeans)
set.seed(88)
# Intertidal experiment:
#   Whole-plot factor (between plots):  Disturbance (None / Scrape)
#   Subplot factor (within plots):      Nutrient addition (Control / +N / +P)
#   Design: 6 whole plots per disturbance; 3 subplots per plot
n_plots <- 6; n_sub <- 3; n_dist <- 2
disturbance <- c("None","Scrape")
nutrients   <- c("Control","+N","+P")

dat <- expand.grid(
  plot_id    = 1:(n_plots*n_dist),
  nutrient   = factor(nutrients)
)
dat$disturbance <- factor(rep(rep(disturbance, each=n_plots), each=n_sub))
# Plot random effect (whole-plot error)
plot_re <- rep(rnorm(n_plots*n_dist, 0, 3), each=n_sub)
# Fixed effects: disturbance reduces biomass; +N increases it; interaction
dist_eff <- ifelse(dat$disturbance=="Scrape", -6, 0)
nutr_eff <- ifelse(dat$nutrient=="+N", 5, ifelse(dat$nutrient=="+P", 2, 0))
int_eff  <- ifelse(dat$disturbance=="Scrape" & dat$nutrient=="+N", -3, 0)
dat$biomass <- 20 + dist_eff + nutr_eff + int_eff + plot_re + rnorm(nrow(dat),0,2)
cat("Split-plot dataset — design structure:\n")
cat("  Whole-plot factor: Disturbance (2 levels, 6 plots each)\n")
cat("  Subplot factor:    Nutrient (3 levels, tested within plots)\n")
print(table(dat$disturbance, dat$nutrient))

---
## Fitting split-plot with LMM (correct approach)

In [ ]:
# The correct split-plot analysis uses LMM with plot as a random effect
# plot_id is the whole-plot unit — its random effect IS the whole-plot error
m_split <- lmer(biomass ~ disturbance * nutrient + (1|plot_id), data=dat)
cat("Split-plot LMM summary:\n")
print(anova(m_split))  # lmerTest gives p-values

cat("\nRandom effects (whole-plot variance component):\n")
print(VarCorr(m_split))

cat("\nInterpretation of F-ratios:\n")
cat("  Disturbance tested against whole-plot variance (plot_id random effect)\n")
cat("  Nutrient and interaction tested against subplot residual variance\n")

In [ ]:
# Estimated marginal means and interaction
em <- emmeans(m_split, ~ disturbance * nutrient)
cat("Estimated means:\n"); print(em)
cat("\nSimple effects: nutrient within each disturbance level:\n")
print(emmeans(m_split, ~ nutrient | disturbance))
print(pairs(emmeans(m_split, ~ nutrient | disturbance), adjust="tukey"))

# Interaction plot
ggplot(as.data.frame(summary(em)),
       aes(nutrient, emmean, colour=disturbance, group=disturbance)) +
  geom_point(size=3) + geom_line(linewidth=1) +
  geom_errorbar(aes(ymin=lower.CL, ymax=upper.CL), width=0.15) +
  labs(title="Split-plot interaction: Disturbance × Nutrient",
       y="Biomass (g)", x="Nutrient treatment") +
  theme_minimal()

In [ ]:
# Comparison: wrong analysis (ignoring whole-plot structure)
m_wrong <- lm(biomass ~ disturbance * nutrient, data=dat)
cat("WRONG — ignoring whole-plot error (pseudoreplication):\n")
print(anova(m_wrong))
cat("\nCORRECT — split-plot LMM:\n")
print(anova(m_split))
cat("\nThe wrong analysis over-estimates df for Disturbance,\n")
cat("giving an anti-conservative test for the whole-plot factor.\n")

---
## Common Pitfalls

**1. Testing the whole-plot factor against the subplot error**
This is pseudoreplication (Hurlbert 1984). The whole-plot factor (e.g., disturbance applied to entire plots) is only replicated at the whole-plot level. Testing it against within-plot (subplot) variance uses far too many degrees of freedom and produces anti-conservative F-ratios.

**2. Confusing which factor is whole-plot vs subplot**
The whole-plot factor is the one applied to the larger unit — the one that determines which plots are independent replicates. Subplot factors are nested within whole plots. Getting this wrong reverses the error term assignment.

**3. Using standard two-way ANOVA instead of split-plot**
`lm(y ~ A * B)` treats all observations as independent. In a split-plot, subplots within the same whole plot are correlated. The correct analysis requires the whole-plot random effect (or the classical split-plot ANOVA with two error strata).

**4. Not checking that whole-plot replication is adequate**
The whole-plot factor is tested with fewer degrees of freedom (number of whole plots minus number of whole-plot treatment levels). A split-plot experiment with only 2–3 whole plots per treatment level has very low power for the whole-plot test, regardless of how many subplots are measured.

**5. Interpreting a significant interaction when whole-plot F is non-significant**
An interaction between a whole-plot and subplot factor is tested at the subplot level. A significant interaction does not salvage a non-significant whole-plot main effect — the two tests are independent.

**6. Forgetting to account for the split-plot structure in power calculations**
Power for the whole-plot factor depends on the number of whole plots (not subplots). Adding more subplots increases power for subplot and interaction tests but not for the whole-plot test.


---
*r_methods_library - Samantha McGarrigle*